In [1]:
#Task [1]
import pandas as pd 

df = pd.read_csv("C:/Users/hp/Downloads/Churn_Modelling.csv")

print(f"The dataset contains {df.shape[0]} rows and {df.shape[1]}columns ")

print("Data types :")
print(df.dtypes)

print("/nMissing values :")
print(df.isnull().sum())

In [ ]:
# Clean and Prepare data (Task [2])
df.drop(['RowNumber' , 'CustomerId' , 'Surname'], axis = 1 , inplace = True)
df ['Gender'] = df['Gender'].map({'Male' : 1 , 'Female' : 0})
df = pd.get_dummies(df , columns = ['Geography'] , drop_first = True)
print("Data after cleaning and encoding:")
print(df.head())

In [ ]:
# Feature scaling Task [3]
from sklearn.preprocessing import StandardScaler
x = df.drop('Exited' , axis = 1 )
y = df['Exited']
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

print("Shape of scaled features :", x_scaled.shape)

In [ ]:
# Task [4]
from sklearn.model_selection import train_test_split
import numpy as np

x_train , x_test , y_train , y_test = train_test_split(
    x_scaled, y,
    test_size = 0.3,
    stratify = y,
    random_state = 42 
)

print("Training set shape :" , x_train.shape)
print("Test set shape :", x_test.shape)

print("/nClass distribution in training set :")
print(np.bincount(y_train))    

print("/nClass distribution in test set :")
print(np.bincount(y_test))    

In [ ]:
#Task [5]
from sklearn.svm import SVC 
from sklearn.metrics import confusion_matrix, classification_report

linear_svm = SVC(kernel = 'linear', C = 1 , random_state = 42)

linear_svm.fit(x_train , y_train)

y_pred_linear = linear_svm.predict(x_test)

print("Confusion Matrix :")
print(confusion_matrix(y_test , y_pred_linear))

print("/nClassification Report :")
print(classification_report(y_test , y_pred_linear , zero_division = 0 ))

In [ ]:
# Task [6]
from sklearn.model_selection import GridSearchCV
param_grid = {'C' : [ 0.1 , 1 , 10 ]}
svc = SVC(kernel='linear' , random_state = 42)
grid_search = GridSearchCV(svc , param_grid , cv = 5 , scoring = 'accuracy')
grid_search.fit(x_train , y_train)

print("Best C Value " , grid_search.best_params_['C'])
print("Best cross-validation accuracy :" , grid_search.best_score_)
print("done")

In [ ]:
# Task [7]
from sklearn.svm import SVC 
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform 

param_dist = {
    'C' : loguniform(1e-2 , 1e2),
    'gamma': loguniform(1e-4 ,1e0)
}

svc_rbf = SVC(kernel = 'rbf' , random_state = 42)

random_search = RandomizedSearchCV(
    svc_rbf,
    param_distributions = param_dist,
    n_iter = 10,
    cv = 5,
    scoring = 'accuracy',
    random_state = 42,
    verbose = 1
)
random_search.fit(x_train , y_train)

print("Best parameters :" , random_search.best_params_)
print("Best Cross-Validation Accuracy :" , random_search.best_score_)

In [ ]:
# Task[8]
from sklearn.metrics import confusion_matrix , classification_report
best_rbf_model = random_search.best_estimator_
y_pred_rbf = best_rbf_model.predict(x_test)

from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(best_rbf_model, x_test, y_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rbf))

print("/nClassification Report :")
print(classification_report(y_test , y_pred_rbf))

In [ ]:
# Task [9]
from sklearn.model_selection import cross_val_score
best_rbf_model = random_search.best_estimator_
cv_scores = cross_val_score(best_rbf_model, x_scaled, y, cv = 10, scoring = 'accuracy')

print("Cross-validation Scores :", cv_scores)
print("Average Accuracy :", cv_scores.mean())